In [6]:
using Pkg
Pkg.add(["POMDPs", "POMDPModels", "POMDPTools", "QMDP"])
using POMDPs, POMDPModels, POMDPTools, QMDP

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


# VERİ

In [7]:
const TERCIH_A = 1
const TERCIH_B = 2
const YOKLA    = 1
const SUN_A    = 2
const SUN_B    = 3
const EGILIM_A = 1
const EGILIM_B = 2

struct TercihPOMDP <: POMDP{Int,Int,Int}
    dogruluk::Float64
    TercihPOMDP(dogruluk=0.90) = new(dogruluk)
end

POMDPs.states(m::TercihPOMDP)       = [TERCIH_A, TERCIH_B]
POMDPs.actions(m::TercihPOMDP)      = [YOKLA, SUN_A, SUN_B]
POMDPs.observations(m::TercihPOMDP) = [EGILIM_A, EGILIM_B]
POMDPs.discount(m::TercihPOMDP)     = 0.95
POMDPs.stateindex(m::TercihPOMDP, s)  = s
POMDPs.actionindex(m::TercihPOMDP, a) = a
POMDPs.obsindex(m::TercihPOMDP, o)    = o

function POMDPs.transition(m::TercihPOMDP, s, a)
    if a == YOKLA
        return Deterministic(s)
    else
        return Uniform([TERCIH_A, TERCIH_B])
    end
end

function POMDPs.observation(m::TercihPOMDP, a, sp)
    if a == YOKLA
        if sp == TERCIH_A
            return SparseCat([EGILIM_A, EGILIM_B], [m.dogruluk, 1.0 - m.dogruluk])
        else
            return SparseCat([EGILIM_B, EGILIM_A], [m.dogruluk, 1.0 - m.dogruluk])
        end
    else
        return Uniform([EGILIM_A, EGILIM_B])
    end
end

function POMDPs.reward(m::TercihPOMDP, s, a)
    if a == YOKLA
        return -1.0
    elseif a == SUN_A
        return s == TERCIH_A ? 10.0 : -20.0
    else
        return s == TERCIH_B ? 10.0 : -20.0
    end
end

POMDPs.initialstate(m::TercihPOMDP) = Uniform([TERCIH_A, TERCIH_B])

pomdp = TercihPOMDP()

TercihPOMDP(0.9)

# MODEL

## HAZIRLIK

In [8]:
# ····················· PROBLEM ·····················
S = states(pomdp)
println("S=$S  A=$(actions(pomdp))  O=$(observations(pomdp))  γ=$(discount(pomdp))")

# ····················· ÇÖZÜM ·····················
solver = QMDPSolver()

# ····················· POLİTİKA ·····················
policy = solve(solver, pomdp)

S=[1, 2]  A=[1, 2, 3]  O=[1, 2]  γ=0.95


AlphaVectorPolicy{TercihPOMDP, Int64}(TercihPOMDP(0.9), 2, [[188.7727656633933, 188.7803834296657], [199.7765745465295, 169.78406462443436], [169.7765745465295, 199.78406462443436]], [1, 2, 3])

## UYGULAMA

In [9]:
# ····················· İNANÇ ·····················
up = updater(policy)
b = initialize_belief(up, initialstate(pomdp))
s = rand(initialstate(pomdp))

for t in 1:20
    # ················· POLİTİKA ·················
    a = action(policy, b)
    # ················· EYLEM ·················
    r = reward(pomdp, s, a)
    sp = rand(transition(pomdp, s, a))
    # ················· GÖZLEM ·················
    o = rand(observation(pomdp, a, sp))
    # ················· İNANÇ GÜNCELLEYİCİSİ ·················
    b = update(up, b, a, o)
    s = sp
    println("t=$t  b=$([round(pdf(b,x), digits=2) for x in S])  a=$a  o=$o  r=$r")
end

t=1  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=2  b=[0.5, 0.5]  a=2  o=1  r=10.0
t=3  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=4  b=[0.5, 0.5]  a=2  o=2  r=10.0
t=5  b=[0.1, 0.9]  a=1  o=2  r=-1.0
t=6  b=[0.5, 0.5]  a=3  o=2  r=10.0
t=7  b=[0.1, 0.9]  a=1  o=2  r=-1.0
t=8  b=[0.5, 0.5]  a=3  o=2  r=10.0
t=9  b=[0.1, 0.9]  a=1  o=2  r=-1.0
t=10  b=[0.5, 0.5]  a=3  o=2  r=10.0
t=11  b=[0.1, 0.9]  a=1  o=2  r=-1.0
t=12  b=[0.5, 0.5]  a=3  o=2  r=10.0
t=13  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=14  b=[0.5, 0.5]  a=2  o=2  r=10.0
t=15  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=16  b=[0.5, 0.5]  a=2  o=2  r=10.0
t=17  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=18  b=[0.5, 0.5]  a=2  o=1  r=10.0
t=19  b=[0.9, 0.1]  a=1  o=1  r=-1.0
t=20  b=[0.5, 0.5]  a=2  o=2  r=-20.0


# DEĞERLENDİRME

In [10]:
sim = RolloutSimulator(max_steps=50)
R = [simulate(sim, pomdp, policy, up, initialize_belief(up, initialstate(pomdp))) for _ in 1:1000]
println("Ortalama iskontolu ödül: ", sum(R)/length(R))

Ortalama iskontolu ödül: 53.33542828479905
